# Bank X — Real-time Fraud Monitoring Dashboard

Reads Spark Streaming outputs from `/workspace/data/tp5/output`.

- **Last 20 users** seen in recent transactions
- **Last 10 seconds** of activity
- Windowed metrics (3h / 7d / 3w / 3mo) + lifetime aggregates
- Auto-refresh every 5 seconds

In [1]:
import sys
import importlib
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display
import ipywidgets as widgets

warnings.filterwarnings("ignore")

sys.path.insert(0, "/home/jovyan/work/tp5")
import dashboard_io as dio
importlib.reload(dio)
from dashboard_io import read_latest_parquet, flatten_window_columns

# #region agent log
dio._dbg("A", "notebook_cell1_import", {"flatten_module": dio.flatten_window_columns.__module__})
# #endregion

OUTPUT_BASE = Path("/workspace/data/tp5/output")
RECENT_PATH = OUTPUT_BASE / "recent_transactions"
LIFETIME_PATH = OUTPUT_BASE / "lifetime"
WINDOW_PATHS = {
    "3_hours": OUTPUT_BASE / "windowed" / "3_hours",
    "7_days": OUTPUT_BASE / "windowed" / "7_days",
    "3_weeks": OUTPUT_BASE / "windowed" / "3_weeks",
    "3_months": OUTPUT_BASE / "windowed" / "3_months",
}
REFRESH_SECONDS = 5

In [2]:
# read_latest_parquet + flatten_window_columns imported from tp5/dashboard_io.py (cell 1)


def anomaly_score(row) -> float:
    """Simple heuristic: high amount vs rolling average."""
    avg = row.get("lifetime_avg_amount", row.get("avg_amount", 0)) or 1
    amt = row.get("amount", row.get("lifetime_avg_amount", 0))
    return float(amt) / float(avg) if avg else 0.0


def style_anomalies(df: pd.DataFrame, amount_col: str = "amount") -> object:
    if df.empty:
        return df
    styled = df.style
    if amount_col in df.columns and "lifetime_avg_amount" in df.columns:
        def highlight(row):
            score = anomaly_score(row)
            if score >= 5:
                return ["background-color: #ffcccc"] * len(row)
            if score >= 2:
                return ["background-color: #fff3cd"] * len(row)
            return [""] * len(row)
        styled = styled.apply(highlight, axis=1)
    return styled

In [3]:
def render_dashboard():
    now = datetime.now(timezone.utc)
    cutoff = now - timedelta(seconds=10)

    recent = read_latest_parquet(RECENT_PATH, limit_files=50)
    lifetime = read_latest_parquet(LIFETIME_PATH, limit_files=10)

    display(widgets.HTML(f"<h2>Dashboard @ {now.isoformat()}</h2>"))

    if recent.empty:
        display(widgets.HTML("<p><b>Waiting for data...</b> Start stack: <code>docker compose up -d</code></p>"))
        return

    recent["event_time"] = pd.to_datetime(recent["event_time"], utc=True)
    if "ingested_at" in recent.columns:
        recent["ingested_at"] = pd.to_datetime(recent["ingested_at"], utc=True)
        time_col = "ingested_at"
    else:
        time_col = "event_time"

    last_10s = recent[recent[time_col] >= cutoff].copy()
    if last_10s.empty:
        last_10s = recent.nlargest(30, time_col).copy()

    last_users = (
        recent.sort_values(time_col, ascending=False)["user_id"]
        .drop_duplicates()
        .head(20)
        .tolist()
    )

    display(widgets.HTML(f"<h3>Last 10 seconds activity (by {time_col})</h3>"))
    cols = ["event_time", "user_id", "direction", "counterparty", "amount", "tx_id"]
    if "ingested_at" in last_10s.columns:
        cols = ["ingested_at"] + cols
    show_10s = last_10s[cols].sort_values(time_col, ascending=False) if not last_10s.empty else pd.DataFrame(columns=cols)
    display(style_anomalies(show_10s.head(50)))

    display(widgets.HTML("<h3>Last 20 active users</h3>"))
    display(pd.DataFrame({"user_id": last_users}))

    user_recent = recent[recent["user_id"].isin(last_users)].copy()
    if not lifetime.empty and not user_recent.empty:
        if "processed_at" in lifetime.columns:
            lifetime = lifetime.sort_values("processed_at", ascending=False).drop_duplicates(
                subset=["user_id", "direction"], keep="first"
            )
        merged = user_recent.merge(lifetime, on=["user_id", "direction"], how="left")
        display(widgets.HTML("<h3>Recent txs + lifetime metrics (anomaly colors)</h3>"))
        display(style_anomalies(merged.sort_values(time_col, ascending=False).head(40)))

    for win_name, path in WINDOW_PATHS.items():
        wdf = flatten_window_columns(read_latest_parquet(path, limit_files=15))
        if wdf.empty:
            continue
        wdf = wdf[wdf["user_id"].isin(last_users)] if "user_id" in wdf.columns else wdf
        display(widgets.HTML(f"<h3>Window: {win_name}</h3>"))
        sort_col = "processed_at" if "processed_at" in wdf.columns else wdf.columns[0]
        display(wdf.sort_values(sort_col, ascending=False).head(30))

    if not last_10s.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        last_10s.groupby("direction")["amount"].sum().plot(kind="bar", ax=ax, color=["#2ecc71", "#3498db"])
        ax.set_title("Total amount in last 10 seconds by direction")
        ax.set_ylabel("MRU")
        plt.tight_layout()
        plt.show()

In [ ]:
import time

auto = widgets.Checkbox(value=True, description="Auto-refresh (5s)")
display(auto)

try:
    while auto.value:
        clear_output(wait=True)
        display(auto)
        render_dashboard()
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("Stopped auto-refresh.")

Checkbox(value=True, description='Auto-refresh (5s)')

HTML(value='<h2>Dashboard @ 2026-05-21T14:09:36.062413+00:00</h2>')

HTML(value='<h3>Last 10 seconds activity (by ingested_at)</h3>')

,ingested_at,event_time,user_id,direction,counterparty,amount,tx_id
0,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:09+00:00,user_b_3924,received,client_3490,1.160000,61a9c748-8a6a-4692-b2d4-59b0bf71e3e1
1,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:12+00:00,user_b_2126,received,user_a_388,12.340000,64b4e273-686f-4266-80fa-6fcf52f9969e
28,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:25+00:00,client_4448,sent,client_2944,2.180000,97b59a74-e596-4972-bc84-a8f92557fac6
27,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:22+00:00,user_b_3826,sent,user_a_122,198.710000,6a7d7510-ab59-4f4a-be05-33633fe6dd6b
26,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:16+00:00,client_3292,sent,client_2670,18.690000,a298c972-2af5-441e-98bf-c8622ec8a678
25,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:15+00:00,user_a_2148,sent,user_a_1225,167.790000,451c73f7-822c-4370-9999-f59c4717d76b
24,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:15+00:00,user_a_2148,sent,client_2287,23.630000,bb7854ff-6371-489a-bc1d-151fa385f4e4
23,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:14+00:00,user_b_1865,sent,user_b_1409,3.190000,9130a5b6-c5e6-44e7-a3db-2fbfc20cc008
22,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:12+00:00,user_a_388,sent,user_b_2126,12.340000,64b4e273-686f-4266-80fa-6fcf52f9969e
21,2026-05-21 14:08:46.704000+00:00,2026-05-21 14:08:09+00:00,client_3490,sent,user_b_3924,1.160000,61a9c748-8a6a-4692-b2d4-59b0bf71e3e1


HTML(value='<h3>Last 20 active users</h3>')

,user_id
0,user_b_3924
1,user_a_197
2,user_a_4944
3,client_4938
4,user_b_2422
5,user_a_1916
6,user_a_2897
7,user_b_3501
8,client_2058
9,client_4461


HTML(value='<h3>Window: 3_hours</h3>')

,user_id,direction,avg_amount,tx_count,total_amount,distinct_counterparties,window_name,processed_at,window_start,window_end
2148,user_b_2422,received,5.86,1,5.86,1,3_hours,2026-05-21 13:35:53.188,1779359100000000000,1779369900000000000
2373,user_b_3501,received,3.18,1,3.18,1,3_hours,2026-05-21 13:35:53.188,1779359100000000000,1779369900000000000
2461,user_b_3501,received,3.18,1,3.18,1,3_hours,2026-05-21 13:35:53.188,1779359040000000000,1779369840000000000
3669,user_b_2422,received,5.86,1,5.86,1,3_hours,2026-05-21 13:35:53.188,1779359040000000000,1779369840000000000
